In [5]:
import sys
sys.path.insert(0, '..')

import pandas as pd

import nltk
from nltk.corpus import sentiwordnet as swn
from nltk.corpus import wordnet as wn

from sklearn.metrics import classification_report, f1_score

from utils import normalize_text

# Load the SAME test set used by Naive Bayes and RoBERTa for apples-to-apples comparison
lex_df = pd.read_csv('../data/test_data.csv')

lex_df = lex_df.dropna(subset=['appName', 'content', 'Sentiment']).copy()
lex_df['content'] = lex_df['content'].apply(normalize_text)
lex_df = lex_df[lex_df['content'] != ''].reset_index(drop=True)

print('Rows:', len(lex_df))
print('Sentiment counts:', lex_df['Sentiment'].value_counts().to_dict())

# Download required NLTK resources if missing
for pkg in ['wordnet', 'sentiwordnet', 'omw-1.4']:
    try:
        nltk.data.find(f'corpora/{pkg}')
    except LookupError:
        nltk.download(pkg)

Rows: 4105
Sentiment counts: {'Negative': 1637, 'Neutral': 1292, 'Positive': 1176}


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/jakobaune/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/jakobaune/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [6]:
# SentiWordNet baseline

import spacy

nlp = spacy.load('en_core_web_sm')

WN_POS = {
    'NOUN': wn.NOUN,
    'VERB': wn.VERB,
    'ADJ': wn.ADJ,
    'ADV': wn.ADV,
}

def sentiwordnet_score(text: str) -> float:
    doc = nlp(text)
    score = 0.0
    hits = 0
    for t in doc:
        if t.is_stop or t.is_punct or t.is_space:
            continue
        wn_pos = WN_POS.get(t.pos_)
        if wn_pos is None:
            continue
        synsets = wn.synsets(t.lemma_, pos=wn_pos)
        if not synsets:
            continue
        try:
            swn_syn = swn.senti_synset(synsets[0].name())
        except Exception:
            continue
        score += (swn_syn.pos_score() - swn_syn.neg_score())
        hits += 1
    return score / hits if hits else 0.0

def score_to_label(s: float, pos_thr=0.05, neg_thr=-0.05):
    if s >= pos_thr:
        return 'Positive'
    if s <= neg_thr:
        return 'Negative'
    return 'Neutral'

lex_df['swn_score'] = lex_df['content'].apply(sentiwordnet_score)
lex_df['swn_pred'] = lex_df['swn_score'].apply(score_to_label)

print('SentiWordNet baseline on shared test set (n=%d)' % len(lex_df))
print('Macro F1:', f1_score(lex_df['Sentiment'], lex_df['swn_pred'], average='macro'))
print(classification_report(lex_df['Sentiment'], lex_df['swn_pred']))

SentiWordNet baseline on shared test set (n=4105)
Macro F1: 0.44228686860887273
              precision    recall  f1-score   support

    Negative       0.71      0.28      0.40      1637
     Neutral       0.34      0.47      0.40      1292
    Positive       0.45      0.64      0.53      1176

    accuracy                           0.44      4105
   macro avg       0.50      0.46      0.44      4105
weighted avg       0.52      0.44      0.44      4105

